# 04 — Advanced Models: XGBoost, LightGBM, LSTM

Trains gradient-boosted trees and the LSTM sequence model with MLflow tracking.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import mlflow
from pathlib import Path

ROOT = Path('..')

In [ ]:
from src.data.fetch_data import fetch_entsoe_load, fetch_openmeteo_weather
from src.data.preprocess import build_features, train_test_split_temporal

load    = fetch_entsoe_load()
weather = fetch_openmeteo_weather()
df      = build_features(load, weather)
train, test = train_test_split_temporal(df)

HORIZON = 7 * 24
X_test  = test.drop(columns=['load_MW'])
y_test  = test['load_MW']

## 1. XGBoost (quantile regression)

In [ ]:
from src.models.ml_models import XGBoostForecaster
from src.evaluation.metrics import evaluate_forecasts

with mlflow.start_run(run_name='xgboost_nb'):
    xgb_model = XGBoostForecaster()
    xgb_model.fit(train)
    xgb_preds = xgb_model.predict(X_test.iloc[:HORIZON])
    xgb_metrics = evaluate_forecasts(
        y_test.iloc[:HORIZON], xgb_preds['forecast'],
        lower_80=xgb_preds['lower_80'], upper_80=xgb_preds['upper_80'],
    )
    mlflow.log_metrics(xgb_metrics)
    print('XGBoost metrics:', xgb_metrics)

## 2. LightGBM (quantile regression)

In [ ]:
from src.models.ml_models import LightGBMForecaster

with mlflow.start_run(run_name='lightgbm_nb'):
    lgbm_model = LightGBMForecaster()
    lgbm_model.fit(train)
    lgbm_preds = lgbm_model.predict(X_test.iloc[:HORIZON])
    lgbm_metrics = evaluate_forecasts(
        y_test.iloc[:HORIZON], lgbm_preds['forecast'],
        lower_80=lgbm_preds['lower_80'], upper_80=lgbm_preds['upper_80'],
    )
    mlflow.log_metrics(lgbm_metrics)
    print('LightGBM metrics:', lgbm_metrics)

## 3. LSTM

In [ ]:
from src.models.deep_learning import LSTMForecaster

with mlflow.start_run(run_name='lstm_nb'):
    lstm_model = LSTMForecaster(input_size=train.shape[1], horizon=HORIZON)
    lstm_model.fit(train, epochs=30)
    lstm_raw  = lstm_model.predict(test)
    lstm_preds = pd.Series(lstm_raw, index=test.index[:HORIZON])
    lstm_metrics = evaluate_forecasts(y_test.iloc[:HORIZON], lstm_preds)
    mlflow.log_metrics(lstm_metrics)
    print('LSTM metrics:', lstm_metrics)

## 4. Model comparison

In [ ]:
from src.visualization.plots import model_comparison_plotly

results = pd.DataFrame([
    {'model': 'XGBoost',  **xgb_metrics},
    {'model': 'LightGBM', **lgbm_metrics},
    {'model': 'LSTM',     **lstm_metrics},
])

fig = model_comparison_plotly(results)
fig.show()
results